# Parte 3: Modelos de Machine Learning

**Disciplina**: Big Data  
**Referencia**: Baldi, P., Sadowski, P. & Whiteson, D. *Searching for exotic particles in high-energy physics with deep learning*. Nature Communications 5, 4308 (2014).

Este notebook treina tres modelos de classificacao sobre o dataset SUSY e compara o desempenho usando dois conjuntos de features:

- **8 low-level**: medicoes brutas do detector (momento, angulos, energia perdida)
- **18 todas**: as 8 acima mais as 10 variaveis derivadas manualmente por fisicos

O experimento replica a pergunta central do paper: uma rede neural consegue aprender as representacoes que fisicos derivaram manualmente?  
Benchmark do paper (deep learning): AUC 0.876 com 8 features, AUC 0.885 com 18 features.

## Setup

Carregamos as bibliotecas do Spark ML e iniciamos a sessao. O ponto de entrada e o Parquet gerado na Parte 2, que contem os dados ja limpos e balanceados sem precisar reler o CSV de 1.61 GB.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (DecisionTreeClassifier,
                                        LogisticRegression,
                                        MultilayerPerceptronClassifier)
from pyspark.ml.evaluation import (MulticlassClassificationEvaluator,
                                    BinaryClassificationEvaluator)
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

spark = SparkSession.builder \
    .appName("SUSY-AC2-Modelos") \
    .getOrCreate()

print(f"Spark {spark.version} pronto.")

Spark 3.5.0 pronto.


In [2]:
df = spark.read.parquet("./data/susy_parquet")

print(f"Linhas carregadas: {df.count():,}")
df.printSchema()

Linhas carregadas: 4,575,763
root
 |-- label: integer (nullable = true)
 |-- lepton1_pT: double (nullable = true)
 |-- lepton1_eta: double (nullable = true)
 |-- lepton1_phi: double (nullable = true)
 |-- lepton2_pT: double (nullable = true)
 |-- lepton2_eta: double (nullable = true)
 |-- lepton2_phi: double (nullable = true)
 |-- missing_energy_magnitude: double (nullable = true)
 |-- missing_energy_phi: double (nullable = true)
 |-- MET_rel: double (nullable = true)
 |-- axial_MET: double (nullable = true)
 |-- M_R: double (nullable = true)
 |-- M_TR_2: double (nullable = true)
 |-- R: double (nullable = true)
 |-- MT2: double (nullable = true)
 |-- S_R: double (nullable = true)
 |-- M_Delta_R: double (nullable = true)
 |-- dPhi_r_b: double (nullable = true)
 |-- cos_theta_r1: double (nullable = true)



## 3.1. Definicao dos Conjuntos de Features

Os dois conjuntos replicam as configuracoes experimentais do paper. Cada modelo sera treinado duas vezes, uma para cada conjunto, permitindo comparar diretamente o impacto das features derivadas.

In [3]:
low_level_cols = [
    "lepton1_pT", "lepton1_eta", "lepton1_phi",
    "lepton2_pT", "lepton2_eta", "lepton2_phi",
    "missing_energy_magnitude", "missing_energy_phi"
]

all_cols = [c for c in df.columns if c != "label"]

feature_sets = {
    "8 low-level": low_level_cols,
    "18 todas"   : all_cols,
}

print(f"Conjunto low-level ({len(low_level_cols)} features): {low_level_cols}")
print(f"Conjunto completo  ({len(all_cols)} features): {all_cols}")

Conjunto low-level (8 features): ['lepton1_pT', 'lepton1_eta', 'lepton1_phi', 'lepton2_pT', 'lepton2_eta', 'lepton2_phi', 'missing_energy_magnitude', 'missing_energy_phi']
Conjunto completo  (18 features): ['lepton1_pT', 'lepton1_eta', 'lepton1_phi', 'lepton2_pT', 'lepton2_eta', 'lepton2_phi', 'missing_energy_magnitude', 'missing_energy_phi', 'MET_rel', 'axial_MET', 'M_R', 'M_TR_2', 'R', 'MT2', 'S_R', 'M_Delta_R', 'dPhi_r_b', 'cos_theta_r1']


## 3.2. Divisao Treino e Teste

Um unico split 80/20 e aplicado sobre os dados brutos do Parquet. Os dois experimentos (8 e 18 features) usam exatamente os mesmos conjuntos de treino e teste, garantindo comparacao justa.

O `seed=42` assegura reproducibilidade: qualquer reexecucao produz a mesma divisao.

In [4]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

total = train_df.count() + test_df.count()
print(f"Treino : {train_df.count():,} ({train_df.count()/total:.0%})")
print(f"Teste  : {test_df.count():,} ({test_df.count()/total:.0%})")

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_com

Py4JError: An error occurred while calling o34.count

## 3.3. Pipeline de Treino e Avaliacao

A funcao `treinar_modelo` monta um `Pipeline` com tres estagios:

1. `VectorAssembler`: concatena as colunas selecionadas em um vetor `"features"`
2. `StandardScaler`: normaliza para media zero e desvio padrao 1
3. Modelo: recebe `"scaled_features"` como entrada

O `StandardScaler` e aplicado em todos os modelos, incluindo a Arvore de Decisao. A decisao e baseada na EDA da Parte 2: as features tem escalas muito diferentes (`lepton_pT` ate ~20, `phi` entre -pi e pi, `cos_theta_r1` entre 0 e 1). Normalizar nao prejudica a Arvore (que usa apenas comparacoes relativas para definir splits) e garante um Pipeline unico e consistente para os tres modelos.

In [ ]:
def treinar_modelo(model, feature_cols):
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    scaler    = StandardScaler(inputCol="features", outputCol="scaled_features",
                               withMean=True, withStd=True)
    pipeline  = Pipeline(stages=[assembler, scaler, model])
    fitted    = pipeline.fit(train_df)
    preds     = fitted.transform(test_df)

    acc = MulticlassClassificationEvaluator(
              labelCol="label", metricName="accuracy").evaluate(preds)
    f1  = MulticlassClassificationEvaluator(
              labelCol="label", metricName="f1").evaluate(preds)
    auc = BinaryClassificationEvaluator(
              labelCol="label", metricName="areaUnderROC").evaluate(preds)

    return {"Accuracy": round(acc, 4), "F1": round(f1, 4), "AUC-ROC": round(auc, 4)}, preds

## 3.4. Arvore de Decisao

A Arvore de Decisao aprende uma sequencia de regras "se/entao" sobre os valores das features, particionando o espaco de dados recursivamente. E o modelo mais interpretavel dos tres: e possivel inspecionar as regras aprendidas.

Parametros:
- `maxDepth=10`: limita a profundidade da arvore para evitar overfitting
- `seed=42`: reproducibilidade das escolhas aleatorias internas
- Embora a Arvore de Decisao nao seja sensivelmente afetada pela escala das features, ela recebe `scaled_features` por consistencia com os outros modelos no Pipeline.

In [ ]:
resultados_dt = {}
preds_dt = {}

dt = DecisionTreeClassifier(labelCol="label", featuresCol="scaled_features",
                             maxDepth=10, seed=42)

for nome_feat, feat_cols in feature_sets.items():
    print(f"Treinando Arvore de Decisao | {nome_feat}...")
    metricas, preds = treinar_modelo(dt, feat_cols)
    resultados_dt[nome_feat] = metricas
    preds_dt[nome_feat] = preds
    print(f"  Accuracy: {metricas['Accuracy']:.4f} | F1: {metricas['F1']:.4f} | AUC-ROC: {metricas['AUC-ROC']:.4f}")

## 3.5. Regressao Logistica

A Regressao Logistica e um modelo linear: aprende um peso para cada feature e usa a funcao sigmoidepara converter a combinacao linear em probabilidade. E sensivelmente afetada pela escala das features, por isso o `StandardScaler` e especialmente importante aqui.

Parametros:
- `maxIter=100`: numero maximo de iteracoes do otimizador (LBFGS)
- `regParam=0.01`: regularizacao L2 para evitar overfitting

In [ ]:
resultados_lr = {}
preds_lr = {}

lr = LogisticRegression(labelCol="label", featuresCol="scaled_features",
                        maxIter=100, regParam=0.01)

for nome_feat, feat_cols in feature_sets.items():
    print(f"Treinando Regressao Logistica | {nome_feat}...")
    metricas, preds = treinar_modelo(lr, feat_cols)
    resultados_lr[nome_feat] = metricas
    preds_lr[nome_feat] = preds
    print(f"  Accuracy: {metricas['Accuracy']:.4f} | F1: {metricas['F1']:.4f} | AUC-ROC: {metricas['AUC-ROC']:.4f}")

## 3.6. Rede Neural (MultilayerPerceptronClassifier)

O `MultilayerPerceptronClassifier` do PySpark implementa uma rede neural totalmente conectada (feedforward). A lista `layers` define o numero de neuronios em cada camada, incluindo entrada e saida.

**Arquitetura escolhida**: `[n_input, 300, 300, 2]`

O paper Baldi et al. (2014) usa uma rede com 5 camadas ocultas de 300 neuronios cada (totalizando ~300k parametros). Essa configuracao e inviavel no `MultilayerPerceptronClassifier` do PySpark com 4M+ linhas, pois a implementacao nao usa GPU nem mini-batches eficientes. Usamos 2 camadas ocultas de 300 neuronios, que mantem a capacidade por camada do paper com menos profundidade, e e computacionalmente executavel.

O tamanho da camada de entrada muda entre os dois experimentos:
- 8 features: `layers = [8, 300, 300, 2]`
- 18 features: `layers = [18, 300, 300, 2]`

A camada de saida tem sempre 2 neuronios (um por classe).

In [ ]:
resultados_mlp = {}
preds_mlp = {}

for nome_feat, feat_cols in feature_sets.items():
    layers = [len(feat_cols), 300, 300, 2]
    print(f"Treinando Rede Neural MLP | {nome_feat} | arquitetura {layers}...")
    mlp = MultilayerPerceptronClassifier(
        labelCol="label", featuresCol="scaled_features",
        layers=layers, maxIter=100, seed=42
    )
    metricas, preds = treinar_modelo(mlp, feat_cols)
    resultados_mlp[nome_feat] = metricas
    preds_mlp[nome_feat] = preds
    print(f"  Accuracy: {metricas['Accuracy']:.4f} | F1: {metricas['F1']:.4f} | AUC-ROC: {metricas['AUC-ROC']:.4f}")

## 3.7. Avaliacao Comparativa

Consolidamos as metricas dos 6 treinos (3 modelos x 2 feature sets) em uma unica tabela. O verde destaca o melhor valor de AUC-ROC por conjunto de features.

Referencia do paper (deep learning com o dataset completo):
- AUC 0.876 com 8 features low-level
- AUC 0.885 com todas as 18 features

In [ ]:
rows = []
for modelo, resultados in [("Arvore de Decisao",  resultados_dt),
                            ("Regressao Logistica", resultados_lr),
                            ("Rede Neural MLP",     resultados_mlp)]:
    for feat_nome, metricas in resultados.items():
        rows.append({"Modelo": modelo, "Features": feat_nome, **metricas})

comparison = pd.DataFrame(rows).set_index(["Modelo", "Features"])

benchmark = pd.DataFrame([
    {"Modelo": "Benchmark paper (deep learning)", "Features": "8 low-level",
     "Accuracy": "-", "F1": "-", "AUC-ROC": 0.876},
    {"Modelo": "Benchmark paper (deep learning)", "Features": "18 todas",
     "Accuracy": "-", "F1": "-", "AUC-ROC": 0.885},
]).set_index(["Modelo", "Features"])

print("=== Resultados completos ===")
display(pd.concat([comparison, benchmark]).style.highlight_max(
    subset=["AUC-ROC"], color="lightgreen"))

In [ ]:
modelos   = ["Arvore de Decisao", "Regressao Logistica", "Rede Neural MLP"]
auc_low   = [resultados_dt["8 low-level"]["AUC-ROC"],
             resultados_lr["8 low-level"]["AUC-ROC"],
             resultados_mlp["8 low-level"]["AUC-ROC"]]
auc_all   = [resultados_dt["18 todas"]["AUC-ROC"],
             resultados_lr["18 todas"]["AUC-ROC"],
             resultados_mlp["18 todas"]["AUC-ROC"]]

x   = np.arange(len(modelos))
w   = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, auc_low, w, label="8 low-level", color="steelblue")
b2 = ax.bar(x + w/2, auc_all, w, label="18 todas",    color="tomato")

ax.axhline(0.876, color="steelblue", linestyle="--", linewidth=1,
           label="Benchmark paper (8 features, AUC 0.876)")
ax.axhline(0.885, color="tomato",    linestyle="--", linewidth=1,
           label="Benchmark paper (18 features, AUC 0.885)")

ax.set_xticks(x)
ax.set_xticklabels(modelos)
ax.set_ylabel("AUC-ROC")
ax.set_title("AUC-ROC por Modelo e Conjunto de Features")
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=8)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("./docs/auc_comparativo.png", dpi=100, bbox_inches="tight")
plt.show()

## 3.8. Salvar Predicoes

Salvamos as predicoes de cada combinacao modelo x features em Parquet separado, para auditoria e eventuais analises de erro.

In [ ]:
import os
os.makedirs("./output", exist_ok=True)

configs = [
    ("dt",  preds_dt),
    ("lr",  preds_lr),
    ("mlp", preds_mlp),
]

for nome_modelo, preds_dict in configs:
    for feat_nome, preds in preds_dict.items():
        sufixo  = "lowlevel" if feat_nome == "8 low-level" else "all"
        caminho = f"./output/predicoes_{nome_modelo}_{sufixo}"
        preds.select("label", "prediction", "probability") \
             .write.mode("overwrite").parquet(caminho)
        print(f"Salvo: {caminho}")

print("\nTodos os arquivos de predicao gerados.")

## Resumo da Parte 3

| Modelo | Features | Accuracy | F1 | AUC-ROC | Benchmark paper |
|---|---|---|---|---|---|
| Arvore de Decisao | 8 low-level | (ver saida) | (ver saida) | (ver saida) | - |
| Arvore de Decisao | 18 todas | (ver saida) | (ver saida) | (ver saida) | - |
| Regressao Logistica | 8 low-level | (ver saida) | (ver saida) | (ver saida) | - |
| Regressao Logistica | 18 todas | (ver saida) | (ver saida) | (ver saida) | - |
| Rede Neural MLP | 8 low-level | (ver saida) | (ver saida) | (ver saida) | 0.876 |
| Rede Neural MLP | 18 todas | (ver saida) | (ver saida) | (ver saida) | 0.885 |

**Interpretacao esperada**: A diferenca de AUC entre 8 e 18 features deve ser pequena para o MLP (como demonstrado no paper) e potencialmente maior para DT e LR, pois esses modelos lineares/rascos se beneficiam mais das features derivadas que ja capturam a fisica relevante.